# Gemma 2 + RAG + Genetic Algorithm + Quantum Dice + Game Theory

This notebook presents a **hybrid decision-and-generation architecture** that combines:

1. **Gemma 2-ready LLM interface** (with a safe mock fallback for local testing)
2. **RAG** over a small document collection
3. **Genetic Algorithm** for tuning retrieval and generation hyperparameters
4. **Quantum Dice (`qdice`)** for deterministic + probabilistic scenario ranking
5. **Game Theory** for selecting a robust interaction strategy between retrieval and generation

The notebook is designed to be:

- **executable end-to-end**
- **easy to adapt to Colab / local environment**
- **honest about dependencies**: the fully tested path here uses a mock LLM, while the Gemma 2 path is prepared for a real runtime with model access.

---

## Core idea

Classic RAG answers a question using retrieved documents.  
This notebook adds three extra layers:

- **Genetic Algorithm** searches for the best retrieval/generation configuration
- **Quantum Dice** ranks candidate solutions both deterministically and probabilistically
- **Game Theory** chooses a robust operating mode when multiple subsystems have competing incentives

This gives a pipeline that is not just “retrieve and answer,” but also:

- **optimize**
- **compare**
- **hedge**
- **select**
- **explain**


In [1]:
# Optional installs for Colab / local runtime
# Uncomment if needed:
# !pip install -q numpy pandas scikit-learn matplotlib transformers accelerate torch

import math
import random
import textwrap
from dataclasses import dataclass
from typing import List, Dict, Tuple, Callable, Any

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

random.seed(42)
np.random.seed(42)


## 1. Mini knowledge base

To keep the notebook executable without internet downloads, we build a small internal corpus.
In a production version, these documents could come from PDF parsing, a vector database, or enterprise knowledge sources.


In [2]:
documents = [
    {
        "id": "doc_1",
        "title": "RAG basics",
        "text": (
            "Retrieval-Augmented Generation improves factual answering by retrieving relevant documents "
            "before generation. The retriever narrows the context, and the generator produces a response "
            "grounded in evidence."
        ),
    },
    {
        "id": "doc_2",
        "title": "Gemma and lightweight open models",
        "text": (
            "Gemma-family models can be used as compact open-weight language models for local or cloud workflows. "
            "They are useful in instruction-following pipelines, summarization, and grounded question answering."
        ),
    },
    {
        "id": "doc_3",
        "title": "Genetic algorithm for tuning",
        "text": (
            "A genetic algorithm evolves a population of candidate solutions. Selection, crossover, and mutation "
            "can optimize hyperparameters such as top_k retrieval depth, score thresholds, or generation settings."
        ),
    },
    {
        "id": "doc_4",
        "title": "Quantum Dice idea",
        "text": (
            "Quantum Dice evaluates multiple scenarios, chooses the best one deterministically by score, "
            "and also computes a softmax probability distribution over all scenarios, giving both certainty "
            "and controlled uncertainty."
        ),
    },
    {
        "id": "doc_5",
        "title": "Game theory in AI pipelines",
        "text": (
            "Game theory can model competing or cooperating subsystems. A conservative retriever may maximize precision, "
            "while an aggressive generator may maximize coverage. A payoff table helps select robust strategies."
        ),
    },
    {
        "id": "doc_6",
        "title": "Grounding and hallucination",
        "text": (
            "Grounding reduces hallucination by linking output claims to retrieved passages. A grounded answer should "
            "prefer evidence-backed statements over unsupported fluency."
        ),
    },
]

df_docs = pd.DataFrame(documents)
df_docs


,id,title,text
0,doc_1,RAG basics,Retrieval-Augmented Generation improves factua...
1,doc_2,Gemma and lightweight open models,Gemma-family models can be used as compact ope...
2,doc_3,Genetic algorithm for tuning,A genetic algorithm evolves a population of ca...
3,doc_4,Quantum Dice idea,"Quantum Dice evaluates multiple scenarios, cho..."
4,doc_5,Game theory in AI pipelines,Game theory can model competing or cooperating...
5,doc_6,Grounding and hallucination,Grounding reduces hallucination by linking out...


## 2. Vector retrieval with TF-IDF

This is a lightweight stand-in for a vector database.  
It lets us test the architecture without external services.


In [3]:
class SimpleRetriever:
    def __init__(self, docs: List[Dict[str, str]]):
        self.docs = docs
        self.vectorizer = TfidfVectorizer(stop_words="english")
        self.doc_matrix = self.vectorizer.fit_transform([d["text"] for d in docs])

    def retrieve(self, query: str, top_k: int = 3) -> List[Dict[str, Any]]:
        q_vec = self.vectorizer.transform([query])
        sims = cosine_similarity(q_vec, self.doc_matrix)[0]
        ranked_idx = np.argsort(sims)[::-1][:top_k]
        results = []
        for idx in ranked_idx:
            item = dict(self.docs[idx])
            item["score"] = float(sims[idx])
            results.append(item)
        return results

retriever = SimpleRetriever(documents)

query = "How can RAG use genetic algorithms and quantum dice to choose the best answer strategy?"
retriever.retrieve(query, top_k=3)


[{'id': 'doc_4',
  'title': 'Quantum Dice idea',
  'text': 'Quantum Dice evaluates multiple scenarios, chooses the best one deterministically by score, and also computes a softmax probability distribution over all scenarios, giving both certainty and controlled uncertainty.',
  'score': 0.3024874218847503},
 {'id': 'doc_6',
  'title': 'Grounding and hallucination',
  'text': 'Grounding reduces hallucination by linking output claims to retrieved passages. A grounded answer should prefer evidence-backed statements over unsupported fluency.',
  'score': 0.11489047645948437},
 {'id': 'doc_3',
  'title': 'Genetic algorithm for tuning',
  'text': 'A genetic algorithm evolves a population of candidate solutions. Selection, crossover, and mutation can optimize hyperparameters such as top_k retrieval depth, score thresholds, or generation settings.',
  'score': 0.10841017567719037}]

## 3. Quantum Dice

`qdice()` returns:

- the **best scenario** by score
- the **score table**
- the **softmax probability distribution** over all scenarios

This is useful when we want both:
- a single best option
- a nuanced probability ranking


In [4]:
def softmax(x: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    x = np.array(x, dtype=float)
    temperature = max(1e-6, float(temperature))
    z = (x - np.max(x)) / temperature
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z)

def qdice(
    scenarios: List[Any],
    scoring_fn: Callable[[Any], float],
    temperature: float = 1.0,
) -> Dict[str, Any]:
    scored = [{"scenario": s, "score": float(scoring_fn(s))} for s in scenarios]
    scores = np.array([item["score"] for item in scored], dtype=float)
    probs = softmax(scores, temperature=temperature)

    for item, p in zip(scored, probs):
        item["probability"] = float(p)

    best = max(scored, key=lambda x: x["score"])
    return {
        "best": best,
        "ranking": sorted(scored, key=lambda x: x["score"], reverse=True),
        "probabilities": probs,
    }

toy_scenarios = [
    {"name": "config_A", "utility": 0.62},
    {"name": "config_B", "utility": 0.81},
    {"name": "config_C", "utility": 0.74},
]

qdice_result = qdice(
    toy_scenarios,
    scoring_fn=lambda s: s["utility"],
    temperature=0.35,
)
qdice_result["ranking"]


[{'scenario': {'name': 'config_B', 'utility': 0.81},
  'score': 0.81,
  'probability': 0.41669854640671894},
 {'scenario': {'name': 'config_C', 'utility': 0.74},
  'score': 0.74,
  'probability': 0.3411639147060733},
 {'scenario': {'name': 'config_A', 'utility': 0.62},
  'score': 0.62,
  'probability': 0.24213753888720776}]

## 4. Gemma 2-ready generator interface

This cell supports two modes:

- **mock**: fully executable here
- **gemma2**: prepared for a real environment with model access

The mock mode makes the notebook testable.  
The Gemma 2 mode is the deployment-ready path.


In [5]:
MODEL_BACKEND = "mock"   # change to "gemma2" in a real environment

class MockLLM:
    def generate(self, prompt: str, max_new_tokens: int = 180) -> str:
        prompt = " ".join(prompt.split())
        return (
            "SYNTHETIC ANSWER: Based on the retrieved context, a robust pipeline combines "
            "RAG for grounding, a genetic algorithm for tuning retrieval parameters, "
            "Quantum Dice for ranking candidate strategies, and game theory for choosing "
            "a stable interaction mode between retriever and generator."
        )

class Gemma2LLM:
    def __init__(self, model_name: str = "google/gemma-2-2b-it"):
        from transformers import AutoTokenizer, AutoModelForCausalLM
        import torch

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto"
        )

    def generate(self, prompt: str, max_new_tokens: int = 180) -> str:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        outputs = self.model.generate(**inputs, max_new_tokens=max_new_tokens)
        text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return text

def build_llm(backend: str = "mock"):
    if backend == "gemma2":
        try:
            return Gemma2LLM()
        except Exception as e:
            print(f"Gemma2 backend unavailable, falling back to mock. Reason: {e}")
            return MockLLM()
    return MockLLM()

llm = build_llm(MODEL_BACKEND)
type(llm).__name__


'MockLLM'

## 5. Candidate configurations

Each configuration controls retrieval and synthesis behavior.

We define a small search space:
- `top_k`
- `grounding_weight`
- `novelty_weight`
- `temperature`
- `strategy_mode`


In [6]:
STRATEGY_MODES = ["precision", "balanced", "exploration"]

def random_candidate() -> Dict[str, Any]:
    return {
        "top_k": random.choice([1, 2, 3, 4, 5]),
        "grounding_weight": round(random.uniform(0.4, 1.2), 3),
        "novelty_weight": round(random.uniform(0.0, 0.8), 3),
        "temperature": round(random.uniform(0.2, 1.4), 3),
        "strategy_mode": random.choice(STRATEGY_MODES),
    }

population = [random_candidate() for _ in range(8)]
population[:3]


[{'top_k': 1,
  'grounding_weight': 0.42,
  'novelty_weight': 0.22,
  'temperature': 0.468,
  'strategy_mode': 'exploration'},
 {'top_k': 1,
  'grounding_weight': 0.941,
  'novelty_weight': 0.714,
  'temperature': 0.304,
  'strategy_mode': 'balanced'},
 {'top_k': 1,
  'grounding_weight': 0.424,
  'novelty_weight': 0.175,
  'temperature': 0.806,
  'strategy_mode': 'precision'}]

## 6. Fitness function

We need a numerical score for each candidate configuration.

In this notebook, the fitness blends:
- retrieval relevance
- grounding preference
- novelty preference
- strategic prior depending on the mode

This is a simplified but fully executable proxy for a real evaluation pipeline.


In [7]:
def evaluate_candidate(candidate: Dict[str, Any], query: str, retriever: SimpleRetriever) -> Dict[str, Any]:
    results = retriever.retrieve(query, top_k=candidate["top_k"])
    relevance = float(np.mean([r["score"] for r in results])) if results else 0.0
    diversity = len(set(r["id"] for r in results)) / max(1, candidate["top_k"])

    mode_bonus = {
        "precision": 0.08 if candidate["top_k"] <= 2 else -0.03,
        "balanced": 0.05,
        "exploration": 0.08 if candidate["top_k"] >= 4 else -0.02,
    }[candidate["strategy_mode"]]

    utility = (
        candidate["grounding_weight"] * relevance
        + candidate["novelty_weight"] * diversity * 0.3
        + mode_bonus
        - 0.02 * candidate["temperature"]
    )

    return {
        "candidate": candidate,
        "relevance": relevance,
        "diversity": diversity,
        "utility": float(utility),
        "docs": results,
    }

evaluated = evaluate_candidate(random_candidate(), query, retriever)
evaluated


{'candidate': {'top_k': 1,
  'grounding_weight': 0.842,
  'novelty_weight': 0.664,
  'temperature': 0.942,
  'strategy_mode': 'balanced'},
 'relevance': 0.3024874218847503,
 'diversity': 1.0,
 'utility': 0.4850544092269598,
 'docs': [{'id': 'doc_4',
   'title': 'Quantum Dice idea',
   'text': 'Quantum Dice evaluates multiple scenarios, chooses the best one deterministically by score, and also computes a softmax probability distribution over all scenarios, giving both certainty and controlled uncertainty.',
   'score': 0.3024874218847503}]}

## 7. Genetic Algorithm

Now we evolve better configurations.

The GA steps are:
1. initialize population
2. evaluate fitness
3. keep elites
4. crossover
5. mutation
6. repeat


In [8]:
def crossover(a: Dict[str, Any], b: Dict[str, Any]) -> Dict[str, Any]:
    child = {}
    for key in a.keys():
        child[key] = random.choice([a[key], b[key]])
    return child

def mutate(c: Dict[str, Any], mutation_rate: float = 0.2) -> Dict[str, Any]:
    c = dict(c)
    if random.random() < mutation_rate:
        c["top_k"] = random.choice([1, 2, 3, 4, 5])
    if random.random() < mutation_rate:
        c["grounding_weight"] = round(random.uniform(0.4, 1.2), 3)
    if random.random() < mutation_rate:
        c["novelty_weight"] = round(random.uniform(0.0, 0.8), 3)
    if random.random() < mutation_rate:
        c["temperature"] = round(random.uniform(0.2, 1.4), 3)
    if random.random() < mutation_rate:
        c["strategy_mode"] = random.choice(STRATEGY_MODES)
    return c

def run_ga(
    query: str,
    retriever: SimpleRetriever,
    population_size: int = 12,
    generations: int = 8,
    elite_size: int = 3,
    mutation_rate: float = 0.25,
) -> Dict[str, Any]:
    population = [random_candidate() for _ in range(population_size)]
    history = []

    for gen in range(generations):
        evaluated = [evaluate_candidate(c, query, retriever) for c in population]
        evaluated = sorted(evaluated, key=lambda x: x["utility"], reverse=True)
        history.append({
            "generation": gen,
            "best_utility": evaluated[0]["utility"],
            "mean_utility": float(np.mean([e["utility"] for e in evaluated])),
        })

        elites = [e["candidate"] for e in evaluated[:elite_size]]
        new_population = elites[:]

        while len(new_population) < population_size:
            p1, p2 = random.sample(elites + [e["candidate"] for e in evaluated[:6]], 2)
            child = crossover(p1, p2)
            child = mutate(child, mutation_rate=mutation_rate)
            new_population.append(child)

        population = new_population

    final_eval = [evaluate_candidate(c, query, retriever) for c in population]
    final_eval = sorted(final_eval, key=lambda x: x["utility"], reverse=True)

    return {
        "best": final_eval[0],
        "top_results": final_eval[:5],
        "history": pd.DataFrame(history),
    }

ga_result = run_ga(query, retriever)
ga_result["best"]


{'candidate': {'top_k': 1,
  'grounding_weight': 1.158,
  'novelty_weight': 0.693,
  'temperature': 0.203,
  'strategy_mode': 'precision'},
 'relevance': 0.3024874218847503,
 'diversity': 1.0,
 'utility': 0.6341204345425407,
 'docs': [{'id': 'doc_4',
   'title': 'Quantum Dice idea',
   'text': 'Quantum Dice evaluates multiple scenarios, chooses the best one deterministically by score, and also computes a softmax probability distribution over all scenarios, giving both certainty and controlled uncertainty.',
   'score': 0.3024874218847503}]}

In [9]:
ga_result["history"]


,generation,best_utility,mean_utility
0,0,0.572676,0.297114
1,1,0.611736,0.379320
2,2,0.611736,0.442925
3,3,0.611736,0.531756
4,4,0.634120,0.522515
5,5,0.634120,0.498779
6,6,0.634120,0.543551
7,7,0.634120,0.565790


## 8. Game Theory layer

We model a simple strategic tension between:

- **Retriever**: conservative vs expansive retrieval
- **Generator**: conservative vs creative synthesis

The payoffs are illustrative.  
Goal: identify a stable or at least robust joint strategy.


In [10]:
retriever_strategies = ["conservative", "expansive"]
generator_strategies = ["conservative", "creative"]

# Utility to retriever and generator respectively
payoff_matrix = {
    ("conservative", "conservative"): (0.82, 0.70),
    ("conservative", "creative"):     (0.76, 0.85),
    ("expansive",    "conservative"): (0.68, 0.74),
    ("expansive",    "creative"):     (0.71, 0.88),
}

def best_response_to_generator(g_strategy: str) -> str:
    candidates = [(r, payoff_matrix[(r, g_strategy)][0]) for r in retriever_strategies]
    return max(candidates, key=lambda x: x[1])[0]

def best_response_to_retriever(r_strategy: str) -> str:
    candidates = [(g, payoff_matrix[(r_strategy, g)][1]) for g in generator_strategies]
    return max(candidates, key=lambda x: x[1])[0]

def find_pure_nash():
    equilibria = []
    for r in retriever_strategies:
        for g in generator_strategies:
            r_best = best_response_to_generator(g)
            g_best = best_response_to_retriever(r)
            if r == r_best and g == g_best:
                equilibria.append((r, g, payoff_matrix[(r, g)]))
    return equilibria

nash = find_pure_nash()
nash


[('conservative', 'creative', (0.76, 0.85))]

## 9. Combine GA + Game Theory + Quantum Dice

Now we create several final system scenarios.
Each scenario is scored using:
- GA utility
- strategic robustness
- grounding bias

Then `qdice()` ranks them.


In [11]:
best_ga = ga_result["best"]

def strategy_score(strategy_pair: Tuple[str, str]) -> float:
    r, g = strategy_pair
    return float(np.mean(payoff_matrix[(r, g)]))

final_scenarios = []

for pair in payoff_matrix.keys():
    scenario = {
        "ga_candidate": best_ga["candidate"],
        "strategy_pair": pair,
        "ga_utility": best_ga["utility"],
        "game_score": strategy_score(pair),
        "docs": best_ga["docs"],
    }
    final_scenarios.append(scenario)

def scenario_scoring_fn(s: Dict[str, Any]) -> float:
    return 0.65 * s["ga_utility"] + 0.35 * s["game_score"]

qdice_final = qdice(final_scenarios, scenario_scoring_fn, temperature=0.25)
qdice_final["ranking"]


[{'scenario': {'ga_candidate': {'top_k': 1,
    'grounding_weight': 1.158,
    'novelty_weight': 0.693,
    'temperature': 0.203,
    'strategy_mode': 'precision'},
   'strategy_pair': ('conservative', 'creative'),
   'ga_utility': 0.6341204345425407,
   'game_score': 0.8049999999999999,
   'docs': [{'id': 'doc_4',
     'title': 'Quantum Dice idea',
     'text': 'Quantum Dice evaluates multiple scenarios, chooses the best one deterministically by score, and also computes a softmax probability distribution over all scenarios, giving both certainty and controlled uncertainty.',
     'score': 0.3024874218847503}]},
  'score': 0.6939282824526514,
  'probability': 0.2631228502311264},
 {'scenario': {'ga_candidate': {'top_k': 1,
    'grounding_weight': 1.158,
    'novelty_weight': 0.693,
    'temperature': 0.203,
    'strategy_mode': 'precision'},
   'strategy_pair': ('expansive', 'creative'),
   'ga_utility': 0.6341204345425407,
   'game_score': 0.7949999999999999,
   'docs': [{'id': 'doc_4

## 10. Prompt assembly and answer generation


In [12]:
def build_prompt(user_query: str, retrieved_docs: List[Dict[str, Any]], scenario: Dict[str, Any]) -> str:
    context = "\n\n".join([
        f"[{d['id']}] {d['title']}: {d['text']}" for d in retrieved_docs
    ])
    return f"""
You are a grounded AI assistant.

User query:
{user_query}

Selected strategy:
Retriever = {scenario['strategy_pair'][0]}
Generator = {scenario['strategy_pair'][1]}

Retrieved context:
{context}

Task:
Answer the question using the retrieved evidence.
Explain how RAG, genetic algorithms, Quantum Dice, and game theory fit together.
""".strip()

best_scenario = qdice_final["best"]["scenario"]
prompt = build_prompt(query, best_scenario["docs"], best_scenario)
print(prompt[:1200])


You are a grounded AI assistant.

User query:
How can RAG use genetic algorithms and quantum dice to choose the best answer strategy?

Selected strategy:
Retriever = conservative
Generator = creative

Retrieved context:
[doc_4] Quantum Dice idea: Quantum Dice evaluates multiple scenarios, chooses the best one deterministically by score, and also computes a softmax probability distribution over all scenarios, giving both certainty and controlled uncertainty.

Task:
Answer the question using the retrieved evidence.
Explain how RAG, genetic algorithms, Quantum Dice, and game theory fit together.


In [13]:
answer = llm.generate(prompt, max_new_tokens=220)
print(answer)


SYNTHETIC ANSWER: Based on the retrieved context, a robust pipeline combines RAG for grounding, a genetic algorithm for tuning retrieval parameters, Quantum Dice for ranking candidate strategies, and game theory for choosing a stable interaction mode between retriever and generator.


## 11. Full pipeline wrapper


In [14]:
def hybrid_rag_pipeline(user_query: str, retriever: SimpleRetriever, llm_backend: str = "mock") -> Dict[str, Any]:
    llm = build_llm(llm_backend)
    ga_result = run_ga(user_query, retriever)

    best_ga = ga_result["best"]
    scenarios = []
    for pair in payoff_matrix.keys():
        scenarios.append({
            "ga_candidate": best_ga["candidate"],
            "strategy_pair": pair,
            "ga_utility": best_ga["utility"],
            "game_score": strategy_score(pair),
            "docs": best_ga["docs"],
        })

    ranked = qdice(scenarios, scenario_scoring_fn, temperature=0.25)
    selected = ranked["best"]["scenario"]
    prompt = build_prompt(user_query, selected["docs"], selected)
    answer = llm.generate(prompt, max_new_tokens=220)

    return {
        "query": user_query,
        "selected_candidate": selected["ga_candidate"],
        "selected_strategy_pair": selected["strategy_pair"],
        "retrieved_docs": selected["docs"],
        "answer": answer,
        "ga_history": ga_result["history"],
        "qdice_ranking": ranked["ranking"],
    }

pipeline_result = hybrid_rag_pipeline(
    "Design an AI system that combines Gemma 2, RAG, genetic optimization, quantum dice and game theory.",
    retriever,
    llm_backend=MODEL_BACKEND,
)

pipeline_result["selected_candidate"], pipeline_result["selected_strategy_pair"]


({'top_k': 1,
  'grounding_weight': 1.115,
  'novelty_weight': 0.745,
  'temperature': 0.378,
  'strategy_mode': 'precision'},
 ('conservative', 'creative'))

In [15]:
print("ANSWER:\n")
print(pipeline_result["answer"])

print("\nRETRIEVED DOCS:")
for d in pipeline_result["retrieved_docs"]:
    print(f"- {d['id']} | {d['title']} | score={d['score']:.4f}")


ANSWER:

SYNTHETIC ANSWER: Based on the retrieved context, a robust pipeline combines RAG for grounding, a genetic algorithm for tuning retrieval parameters, Quantum Dice for ranking candidate strategies, and game theory for choosing a stable interaction mode between retriever and generator.

RETRIEVED DOCS:
- doc_4 | Quantum Dice idea | score=0.1841


## 12. Smoke tests

These tests verify that the notebook works end-to-end in the default mode.


In [16]:
assert isinstance(pipeline_result["answer"], str)
assert len(pipeline_result["retrieved_docs"]) >= 1
assert "selected_candidate" in pipeline_result
assert pipeline_result["ga_history"].shape[0] > 0
assert len(pipeline_result["qdice_ranking"]) == 4

print("All smoke tests passed.")


All smoke tests passed.


## 13. How to switch to real Gemma 2

Change:

```python
MODEL_BACKEND = "mock"
```

to:

```python
MODEL_BACKEND = "gemma2"
```

and make sure your environment can access a Gemma 2 checkpoint, for example:

- `google/gemma-2-2b-it`

You may also need:
- authenticated Hugging Face access if the model requires it
- enough RAM / VRAM
- `transformers`, `torch`, `accelerate`

---

## Interpretation of the architecture

### Why RAG?
Because grounding matters.  
RAG narrows the evidence space before generation.

### Why genetic algorithms?
Because many RAG pipelines have fragile hyperparameters.  
Instead of manually guessing `top_k`, weighting, and strategy modes, we evolve them.

### Why Quantum Dice?
Because the highest score is useful, but uncertainty structure is also useful.  
`qdice()` lets us keep both.

### Why game theory?
Because subsystems often have different objectives:
- retriever prefers precision
- generator may prefer expressive coverage

A payoff perspective helps choose a robust equilibrium-like mode.

---

## Production upgrades

This notebook is intentionally lightweight. A production version could replace:
- TF-IDF retriever -> vector DB + dense embeddings
- mock LLM -> real Gemma 2 inference
- toy fitness -> benchmark-based evaluation
- static payoff matrix -> learned or data-driven payoffs
- small corpus -> PDF / database / API knowledge sources
